# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. We will:
- Load and examine dataset metadata from its Croissant schema
- Explore record sets, fields, and columns (referenced by their `@id` values)
- Extract data and perform basic exploratory data analysis (EDA)
- Visualize key dataset features

### Dataset Source
The dataset source is provided via this Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Make sure mlcroissant and pandas are installed
!pip install mlcroissant pandas

## 1. Data Loading
Load the dataset metadata and display a summary description using the `mlcroissant` API. This provides a quick peek at the contents and structure, useful for navigating subsequent steps.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset via the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Show core metadata attributes
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview
Now, enumerate the available record sets and their main fields and columns. We will reference all entities by their `@id` as per the Croissant specification.

> **Note:** For real-world datasets, the `mlcroissant` API makes the record set structure accessible, and their IDs allow exact referencing during extraction and analysis.

In [ ]:
# List and describe available record sets and their fields, using @id for references

# The record_set attribute holds all root-level record sets in the dataset (Croissant 1.0 standard)
# Here we use only record sets with data
record_set_ids = []
record_sets = getattr(meta, 'recordSet', [])
if isinstance(record_sets, dict):
    # If single record set
    record_sets = [record_sets]

print("Record Sets:")
for recset in record_sets:
    @id = getattr(recset, '@id', None) or recset.get('@id')
    print(f"- RecordSet @id: {@id}")
    record_set_ids.append(@id)
    # Show its core fields and their @id/label
    # Fields may be under 'field' property
    fields = getattr(recset, 'field', []) or recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for fld in fields:
        field_id = getattr(fld, '@id', None) or fld.get('@id')
        label = getattr(fld, 'label', None) or fld.get('label', '')
        print(f"    - {field_id}\t(label: {label})")
    # Also show any columns (for tabular record sets)
    columns = getattr(recset, 'column', []) or recset.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("  Columns:")
        for col in columns:
            col_id = getattr(col, '@id', None) or col.get('@id')
            col_label = getattr(col, 'label', None) or col.get('label', '')
            print(f"    - {col_id}\t(label: {col_label})")
print()
# For smaller datasets, let's show an example record structure from each record set
for recset_id in record_set_ids:
    print(f"Example record from record set {@id}: (Reference: {recset_id})")
    try:
        recs = list(dataset.records(record_set=recset_id))
        example_record = recs[0] if recs else None
        print(example_record)
    except Exception as e:
        print(f"  [Error loading records: {e}]")
    print()

## 3. Data Extraction
Let's extract data from each record set, using their precise `@id` values, and load them into Pandas DataFrames for further analysis.

- All columns are referenced by their Croissant `@id`.

In [ ]:
# Extract all record sets and create DataFrames, using @id

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[record_set_id] = df
            print(f"Columns in {record_set_id}:\n", df.columns.tolist())
            display(df.head(3))
        else:
            print("[No records found for this record set]")
    except Exception as e:
        print(f"Failed to load records: {e}")
    print('-' * 60)

# As an example, let's pick one record set for EDA if available
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set for EDA: {example_record_set_id}")
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric column, filter high values, and normalize the numeric data (all columns referenced by their `@id`).

> If unsure about which column is numeric, inspect the DataFrame columns above.

In [ ]:
# Choose a record set and a numeric field @id for EDA
if example_record_set_id:
    df = dataframes[example_record_set_id]
    # Attempt to autodetect a numeric field by checking column types
    # (Columns are referenced by their Croissant @id)
    numeric_candidates = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    if not numeric_candidates:
        # Try parsing some columns to numeric
        import numpy as np
        for col in df.columns:
            try:
                as_numeric = pd.to_numeric(df[col])
                if as_numeric.notnull().sum() > 0 and np.issubdtype(as_numeric.dtype, np.number):
                    numeric_candidates.append(col)
            except Exception:
                pass
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Choose the first detected numeric column
        print(f"Numeric field selected (@id): {numeric_field}")
        # Ensure the column is numeric type
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 1.0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean)")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field
        group_candidates = [c for c in df.columns if df[c].dtype == object]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field (@id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head(10))
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field in the record set. We'll also plot a boxplot by group field if one was selected previously.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and 'numeric_field' in locals():
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping was found
    if 'group_field' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load dataset metadata and records from a Croissant schema using `mlcroissant`
- Reference all dataset entities precisely via their `@id` values, following FAIR best practices
- Explore available record sets and extract tabular data to DataFrames
- Run basic EDA and visualize distributions

You can follow this template to work with other `mlcroissant`-described datasets, simply by updating the Croissant URL and using the presented workflow.

Further analysis (statistical tests, more advanced visualizations, or machine learning workflows) can easily be added in the cells below.